In [1]:
import anthropic
import pathlib
import csv
import pandas as pd
from dotenv import load_dotenv
import re

In [3]:
load_dotenv("../etc/.env")  # loads .env into environment variables

True

In [2]:
client = anthropic.Anthropic()

In [13]:
df = pd.read_csv("exercises_termoquimica.csv")

In [14]:
df.head()

,subject,topic,year,exam,exercise,statement
0,quimica,Termoquímica,2000,Reserva 1,Ejercicio 4 Opción B,a) Dibuje el diagrama de entalpía teniendo en ...
1,química,Termoquímica,2000,Reserva 2,Ejercicio 6 Opción A,"El amoniaco, a 25 “C y 1 atm, se puede oxidar ..."
2,química,Termoquímica,2000,Reserva 3,Ejercicio 5 Opción B,a) Calcule la variación de entalpía estándar c...
3,quimica,Termoquímica,2000,Reserva 4,Ejercicio 5 Opcion A,a) Calcule la variación de entalpía estándar d...
4,química,Termoquímica,2000,Septiembre,Ejercicio 3 Opción B,Indique razonadamente si las siguientes afirma...


In [15]:
df.statement = df.statement.str.replace("\n", " | ")

In [16]:
df_chunks = [df[i:i+50] for i in range(0, len(df), 50) ]

In [ ]:
def clasificar_ejercicios(dataset: pd.DataFrame) -> list[str]:
    
    system_prompt = f""
    dataset.to_csv("data_csv.csv", index=False)
        
    data_text = pathlib.Path("data_csv.csv").read_text()        # subset, no df completo

    # remove the data_csv.csv file after reading its content
    pathlib.Path("data_csv.csv").unlink(missing_ok=True)

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=8192,
        system=system_prompt,
        messages=[{"role": "user",
                    "content": f"Here is the data: \n\n{data_text}"
                    }]
    )

    return response.content[0].text.strip().splitlines()

La respuesta incluye los enunciados, que dan problemas de parseo al convertir a csv. Elimino los enunciados.

In [ ]:
# def remove_statement(line):
#     start, end = [i.start() for i in re.finditer('"', line)]
#     line = line[:start-1] + line[end+1:] 
#     return line

In [38]:
response = clasificar_ejercicios(df_chunks[2])

In [39]:
response[:10]

['```csv',
 'subject,topic,year,exam,exercise,tipo_ejercicio',
 'química,Termoquímica,2014,Reserva 4,Ejercicio 5 Opción A,Entalpías de enlace',
 'química,Termoquímica,2014,Septiembre,Ejercicio 6 Opción A,Ley de Hess',
 'quimica,Termoquímica,2015,Junio,Ejercicio 6 Opción B,Entalpías de formación',
 'química,Termoquímica,2015,Reserva 1,Ejercicio 3 Opción B,Verdadero o Falso',
 'química,Termoquímica,2015,Reserva 2,Ejercicio 6 Opción A,Entalpías de formación',
 'química,Termoquímica,2015,Reserva 3,Ejercicio 5 Opción B,Ley de Hess',
 'química,Termoquímica,2015,Reserva 4,Ejercicio 5 Opción A,Entalpías de enlace',
 'química,Termoquímica,2015,Septiembre,Ejercicio 5 Opción A,Relación entalpía y energía interna']

In [40]:
response = response[3:]

In [41]:
response[:5]

['química,Termoquímica,2014,Septiembre,Ejercicio 6 Opción A,Ley de Hess',
 'quimica,Termoquímica,2015,Junio,Ejercicio 6 Opción B,Entalpías de formación',
 'química,Termoquímica,2015,Reserva 1,Ejercicio 3 Opción B,Verdadero o Falso',
 'química,Termoquímica,2015,Reserva 2,Ejercicio 6 Opción A,Entalpías de formación',
 'química,Termoquímica,2015,Reserva 3,Ejercicio 5 Opción B,Ley de Hess']

In [42]:
classified_data = "\n".join(response)

In [43]:
with open("classified_termoquimica.csv", "a", encoding="utf-8") as file:
    file.writelines(classified_data)

In [44]:
df2 = pd.read_csv("classified_termoquimica.csv", encoding="utf-8")

In [57]:
df2.subject = "química"

In [45]:
df2.sample(10)

,subject,topic,year,exam,exercise,exercise_type
78,química,Termoquímica,2011,Reserva 3,Ejercicio 3 Opción B,Verdadero o Falso
119,química,Termoquímica,2025,Julio,Ejercicio 2A,Energía libre de Gibbs teórico | Cálculo de en...
88,quimica,Termoquímica,2012,Septiembre,Ejercicio 3 Opción B,1er Principio de la Termodinámica
25,quimica,Termoquímica,2004,Junio,Ejercicio 6 Opcióon A,Entalpías de formación
3,quimica,Termoquímica,2000,Reserva 4,Ejercicio 5 Opcion A,Ley de Hess
54,quiímica,Termoquímica,2008,Reserva 1,Ejercicio 3 Opcion B,Verdadero o Falso
19,quimica,Termoquímica,2003,Reserva 1,Ejercicio 5 Opcion A,Entalpías de formación
29,quimica,Termoquímica,2004,Reserva 4,Ejercicio 4 Opción A,Cálculo de entropía
100,química,Termoquímica,2014,Septiembre,Ejercicio 6 Opción A,Ley de Hess
34,quimica,Termoquímica,2005,Reserva 1,Ejercicio 6 Opción B,Entalpías de formación


In [53]:
df2.subject.unique()

<StringArray>
['química']
Length: 1, dtype: str

In [54]:
df2 = df2.sort_values(["year", "exam", "exercise"]).reset_index(drop = True)

In [55]:
df2.to_csv("classified_termoquimica.csv", index = False)

In [5]:
df = pd.read_csv("classified_reactividad_organica.csv")

In [6]:
df.sample(10)

,subject,year,topic,exam,exercise,tipo_ejercicio
8,química,2001,Reactividad Orgánica,Reserva 3,Ejercicio 2 Opción A,Verdadero/falso | Hidrocarburos
104,química,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,Isomería
97,química,2018,Reactividad Orgánica,Reserva 1,Ejercicio 4 Opción B,Reacciones orgánicas | Hidrocarburos
47,química,2009,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería | Reacciones orgánicas
123,química,2021,Reactividad Orgánica,Reserva 2,Ejercicio B6,Verdadero / falso | Isomería | Reacciones orgá...
118,química,2020,Reactividad Orgánica,Reserva 4,Ejercicio B4,Isomería | Reacciones orgánicas
52,química,2010,Reactividad Orgánica,Reserva 1,Ejercicio 4 Opción B,Reacciones orgánicas
27,química,2005,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería
138,química,2024,Reactividad Orgánica,Julio,Ejercicio B6,Reacciones orgánicas | Hidrocarburos
49,química,2009,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,Isomería∫


In [7]:
df.exam.unique()

<StringArray>
[     'Junio',  'Reserva 2',  'Reserva 3',  'Reserva 4', 'Septiembre',
  'Reserva 1',      'Julio']
Length: 7, dtype: str

In [8]:
df.topic.unique()

<StringArray>
['Reactividad Orgánica']
Length: 1, dtype: str

In [11]:
df.tipo_ejercicio.unique()

<StringArray>
[                                    'Reacciones orgánicas',
                                            'Hidrocarburos',
                     'Hidrocarburos | Reacciones orgánicas',
                          'Verdadero/falso | Hidrocarburos',
                                          'Verdadero/falso',
                                                 'Isomería',
                                 'Isomería | Hidrocarburos',
                          'Isomería | Reacciones orgánicas',
                          'Reacciones orgánicas | Isomería',
                               'Verdadero/falso | Isomería',
          'Isomería | Reacciones orgánicas | Hidrocarburos',
                     'Reacciones orgánicas | Hidrocarburos',
        'Verdadero/falso | Isomería | Reacciones orgánicas',
   'Verdadero/falso | Reacciones orgánicas | Hidrocarburos',
      'Verdadero / falso | Isomería | Reacciones orgánicas',
                        'Verdadero / falso | Hidrocarburos',
 'Verdader

In [10]:
df.tipo_ejercicio = df.tipo_ejercicio.str.replace("Isomería∫", "Isomería")